In [4]:
# =========================
# Cell — Diagnostic + HousingStock-weighted hourly kWh (per “typical” consumer)
#           + per-hour ToU tariff columns (current & adjustable)
# =========================
# Inputs (edit these)
DAM_PATH        = r"C:\Users\dmitr\Article_02\DAM_2019-2025_v05.xlsx"
WEIGHTS_PATH    = r"C:\Users\dmitr\Article_02\nrel_weights\weight_solver_15i.xlsx"  # 'archetypes' tab
SUBCLASS_PATH   = r"C:\Users\dmitr\Downloads\Subclass_All_kWh.xlsx"                 # 8,760 rows
OUTPUT_PATH_NEW = r"C:\Users\dmitr\Downloads\Subclass_All_kWh__with_DAM_cols_tou.xlsx"

# Rolling 12m anchor (start of the window)
ANCHOR_DAM_DATE_STR    = "2024-09-29 00:00:00"
HOURS = 8760

# ---- Base rate and ToU multipliers (CURRENT) ----
FLAT_TARIFF_UAH_PER_KWH = 3.6

# 2-zone (current)
CURR_2Z_DAY   = 1.0   # hours 7..22
CURR_2Z_NIGHT = 0.5   # hours 23,0..6

# 3-zone (current)
CURR_3Z_OFF   = 0.4   # hours 23,0..6
CURR_3Z_SEMI  = 1.0   # hours 7, 11..19, 22
CURR_3Z_ON    = 1.5   # hours 8..10, 20..21

# ---- Adjustable multipliers (ADJ) — edit freely ----
ADJ_2Z_DAY   = 1.0
ADJ_2Z_NIGHT = 0.5

ADJ_3Z_OFF   = 0.4
ADJ_3Z_SEMI  = 1.0
ADJ_3Z_ON    = 1.5

import pandas as pd
import numpy as np

def _num_clean(series_like):
    return pd.to_numeric(pd.Series(series_like, dtype="string").str.replace(",", "", regex=False), errors="coerce")

# 1) Load subclass file (do not alter timestamps or order)
sub_df = pd.read_excel(SUBCLASS_PATH)
if len(sub_df) != HOURS:
    raise ValueError(f"Expected {HOURS} rows in {SUBCLASS_PATH}, got {len(sub_df)}.")

# 2) Build the rolling 8,760-hour timeline from the anchor
anchor = pd.to_datetime(ANCHOR_DAM_DATE_STR)
ts_full = pd.date_range(start=anchor, periods=HOURS, freq="H")  # e.g., 2024-09-29 00:00 → 2025-09-28 23:00

# 3) Reorder to match Jan→Dec layout:
years = sorted(ts_full.year.unique())
if len(years) != 2:
    raise ValueError(f"Window spans {len(years)} year(s); expected exactly 2. Got: {years}")

later_year   = max(years)
earlier_year = min(years)

part_later   = ts_full[ts_full.year == later_year]    # e.g., 2025-01-01 00:00 .. 2025-09-28 23:00
part_earlier = ts_full[ts_full.year == earlier_year]  # e.g., 2024-09-29 00:00 .. 2024-12-31 23:00
ts_reordered = part_later.append(part_earlier)         # preserve ascending order within each part

# 4) Map timestamps → (Date, settlement_hour) with rule: hour_of_day = settlement_hour - 1 (00:00→1, 23:00→24)
dates_norm = ts_reordered.normalize()
settlement_hours = (ts_reordered.hour + 1).astype(int)         # 1..24
hour_of_day      = (settlement_hours - 1).astype(int)          # 0..23

seq = pd.DataFrame({"Date": dates_norm, "settlement_hour": settlement_hours})

# 5) Load DAM, keep FIRST occurrence per (Date, settlement_hour)
dam = pd.read_excel(DAM_PATH, sheet_name="DAM")
dam["Date"] = pd.to_datetime(dam["Date"]).dt.normalize()
dam["settlement_hour"] = pd.to_numeric(dam["settlement_hour"], errors="coerce").astype("Int64")
dam = dam.dropna(subset=["Date", "settlement_hour"]).copy()
dam["settlement_hour"] = dam["settlement_hour"].astype(int)

dam["price_uah_per_mwh"] = _num_clean(dam["price_uah_per_mwh"])
dam["volume_sell_mwh"]   = _num_clean(dam["volume_sell_mwh"])

dam_first = dam.drop_duplicates(subset=["Date", "settlement_hour"], keep="first").copy()

# 6) Left-join seq → DAM-first (blanks if missing)
aligned = seq.merge(
    dam_first[["Date", "settlement_hour", "price_uah_per_mwh", "volume_sell_mwh"]],
    on=["Date", "settlement_hour"],
    how="left",
)

# 7) Build enrichment from DAM (leave missing as NaN/NaT so Excel shows blanks)
enrich = pd.DataFrame({
    "DAM_Date": aligned["Date"],
    "settlement_hour": aligned["settlement_hour"].astype("Int64"),
    "DAM_price_uah_per_mwh": aligned["price_uah_per_mwh"],
    "DAM_volume_sell_mwh": aligned["volume_sell_mwh"],
})

# 8) Add HousingStock-weighted average kWh per hour
#    Weighted_kWh_avg_by_HousingStock = sum_i (w_i * Subclass_i_kWh) / sum_i w_i
arch = pd.read_excel(WEIGHTS_PATH, sheet_name="archetypes")
arch = arch.rename(columns={c: c.strip() for c in arch.columns})
arch["sub_class_id"] = arch["sub_class_id"].astype(str)
arch["HousingStock"] = pd.to_numeric(arch["HousingStock"], errors="coerce").fillna(0.0)

subclass_cols = [c for c in sub_df.columns if c.lower().startswith("subclass_") and c.lower().endswith("_kwh")]
if not subclass_cols:
    raise ValueError("No subclass *_kWh columns found in the subclass file.")

subclass_ids = [c.replace("_kWh", "") for c in subclass_cols]
w_map = dict(zip(arch["sub_class_id"], arch["HousingStock"]))
weights = np.array([w_map.get(sid, 0.0) for sid in subclass_ids], dtype=float)
w_sum = float(np.sum(weights))
if w_sum <= 0:
    raise ValueError("Sum of HousingStock weights is zero; cannot form weighted average.")

kwh_matrix = sub_df[subclass_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
weighted_kwh_avg = (kwh_matrix @ weights) / w_sum

enrich["Weighted_kWh_avg_by_HousingStock"] = weighted_kwh_avg

# 9) Compute per-hour ToU tariff columns (current vs adjustable)
#    Hour sets (by hour_of_day = 0..23):
H_2Z_NIGHT = np.array([23] + list(range(0, 7)), dtype=int)   # 23,0..6
H_2Z_DAY   = np.array(list(range(7, 23)), dtype=int)         # 7..22

H_3Z_OFF   = H_2Z_NIGHT                                      # 23,0..6
H_3Z_ON    = np.array(list(range(8, 11)) + list(range(20, 22)), dtype=int)  # 8..10,20..21
H_3Z_SEMI  = np.array([7] + list(range(11, 20)) + [22], dtype=int)         # 7,11..19,22

hod = hour_of_day  # numpy array length 8760

# Helper to build 2-zone rate vector given multipliers
def build_2z_rates(mult_day, mult_night):
    mask_day = np.isin(hod, H_2Z_DAY)
    # vectorized: base * (day*mask + night*(~mask))
    return FLAT_TARIFF_UAH_PER_KWH * (mask_day.astype(float) * mult_day + (~mask_day).astype(float) * mult_night)

# Helper to build 3-zone rate vector given multipliers
def build_3z_rates(mult_off, mult_semi, mult_on):
    # default semi
    rate = np.full_like(hod, FLAT_TARIFF_UAH_PER_KWH * mult_semi, dtype=float)
    rate[np.isin(hod, H_3Z_OFF)] = FLAT_TARIFF_UAH_PER_KWH * mult_off
    rate[np.isin(hod, H_3Z_ON)]  = FLAT_TARIFF_UAH_PER_KWH * mult_on
    return rate

# Current tariff columns
enrich["tou_2z_current"] = build_2z_rates(CURR_2Z_DAY, CURR_2Z_NIGHT)
enrich["tou_3z_current"] = build_3z_rates(CURR_3Z_OFF, CURR_3Z_SEMI, CURR_3Z_ON)

# Adjustable tariff columns
enrich["tou_2z_adj"] = build_2z_rates(ADJ_2Z_DAY, ADJ_2Z_NIGHT)
enrich["tou_3z_adj"] = build_3z_rates(ADJ_3Z_OFF, ADJ_3Z_SEMI, ADJ_3Z_ON)

# 10) Concatenate to the original subclass frame and SAVE
out_df = pd.concat([sub_df.reset_index(drop=True), enrich], axis=1)
out_df.to_excel(OUTPUT_PATH_NEW, index=False)

# Diagnostics
missing_price = out_df["DAM_price_uah_per_mwh"].isna().sum()
missing_vol   = out_df["DAM_volume_sell_mwh"].isna().sum()
print(f"Saved: {OUTPUT_PATH_NEW}")
print(f"Missing price cells (blank): {missing_price}/{HOURS}")
print(f"Missing volume cells (blank): {missing_vol}/{HOURS}")
print(out_df[["DAM_Date","settlement_hour","tou_2z_current","tou_3z_current","tou_2z_adj","tou_3z_adj"]].head(6))

C:\Users\admin\AppData\Local\Temp\ipykernel_23536\47620095.py:36: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  ts_full = pd.date_range(start=anchor, periods=HOURS, freq="H")  # e.g., 2024-09-29 00:00 → 2025-09-28 23:00


Saved: C:\Users\admin\Bess_Valuation\Subclass_All_kWh__with_DAM_cols.xlsx
Missing price cells (blank): 25/8760
Missing volume cells (blank): 25/8760
    DAM_Date  settlement_hour  DAM_price_uah_per_mwh  DAM_volume_sell_mwh  \
0 2025-01-01                1                 3500.0               2705.6   
1 2025-01-01                2                 2900.0               2669.8   
2 2025-01-01                3                 2440.0               2532.8   

   Weighted_kWh_avg_by_HousingStock  
0                          0.198023  
1                          0.256599  
2                          0.234695  


In [10]:
# =========================
# Cell — Subclass cost calc for latest 12m (8,760h) + DAM VWAP + ToU zone costs
#   Source: the enriched file with aligned DAM columns
#   NOTE: No percentage-difference outputs are written.
#   Engine handling: prefer 'xlsxwriter', fallback to 'openpyxl' (like your screenshot).
# =========================
# Inputs (edit these)
ENRICHED_PATH = r"C:\Users\admin\Bess_Valuation\Subclass_All_kWh__with_DAM_cols.xlsx"
OUTPUT_PATH   = r"C:\Users\admin\Bess_Valuation\subclass_costs_summary.xlsx"
FLAT_TARIFF_UAH_PER_KWH = 3.6

# ---- No edits below this line ----
import pandas as pd
import numpy as np

# --- Excel engine picker (prefers xlsxwriter, falls back to openpyxl) ---
def pick_excel_engine(prefer: str | None = None) -> str:
    if prefer:
        try:
            __import__(prefer)
            return prefer
        except Exception:
            pass
    for candidate in ("xlsxwriter", "openpyxl"):
        try:
            __import__(candidate)
            return candidate
        except Exception:
            continue
    raise ModuleNotFoundError(
        "Не знайдено Excel-двигун. Встановіть один із пакетів: "
        "`pip install xlsxwriter` або `pip install openpyxl`."
    )

EXCEL_ENGINE = pick_excel_engine()
print(f"[Cost Cell] Using Excel engine: {EXCEL_ENGINE}")

def _to_num(x):
    return pd.to_numeric(pd.Series(x, dtype="string").str.replace(",", "", regex=False), errors="coerce")

def _weighted_avg_uah_per_kwh(price_uah_per_kwh: np.ndarray,
                              kwh: np.ndarray,
                              hour_of_day: np.ndarray,
                              use_hours: np.ndarray) -> float:
    """Sum(p*q)/Sum(q) over hours with hour∈use_hours and non-NaN price."""
    mask_hours = np.isin(hour_of_day, use_hours)
    mask_price = ~np.isnan(price_uah_per_kwh)
    m = mask_hours & mask_price
    if not np.any(m):
        return np.nan
    q = kwh[m].astype(float)
    p = price_uah_per_kwh[m].astype(float)
    total_q = float(np.sum(q))
    return float(np.sum(p * q) / total_q) if total_q > 0 else np.nan

def _vwap_uah_per_kwh_by_hours(price_uah_per_mwh: np.ndarray,
                               vol_mwh: np.ndarray,
                               hour_of_day: np.ndarray,
                               use_hours: np.ndarray) -> float:
    """Volume-weighted avg price (UAH/kWh) over given hours; requires price & vol>0."""
    m = np.isin(hour_of_day, use_hours) & ~np.isnan(price_uah_per_mwh) & ~np.isnan(vol_mwh) & (vol_mwh > 0)
    if not np.any(m):
        return np.nan
    v = vol_mwh[m].astype(float)
    p = price_uah_per_mwh[m].astype(float)
    vwap_mwh = float(np.sum(p * v) / np.sum(v))
    return vwap_mwh / 1000.0  # to UAH/kWh

# ---------------------------
# 1) Read the enriched file
# ---------------------------
df = pd.read_excel(ENRICHED_PATH)

# Numeric price/volume and derived price in UAH/kWh
if "DAM_price_uah_per_mwh" not in df.columns:
    raise ValueError("Column 'DAM_price_uah_per_mwh' not found in enriched file.")
if "DAM_volume_sell_mwh" not in df.columns:
    raise ValueError("Column 'DAM_volume_sell_mwh' not found in enriched file.")
df["DAM_price_uah_per_mwh"] = _to_num(df["DAM_price_uah_per_mwh"])
df["DAM_volume_sell_mwh"]   = _to_num(df["DAM_volume_sell_mwh"])
df["DAM_price_uah_per_kwh"] = df["DAM_price_uah_per_mwh"] / 1000.0

# Hour-of-day (0..23) from settlement_hour (1..24): hour = sh - 1
if "settlement_hour" not in df.columns:
    raise ValueError("Column 'settlement_hour' not found in enriched file.")
hour_of_day = (pd.to_numeric(df["settlement_hour"], errors="coerce").fillna(1).astype(int) - 1).clip(0, 23).to_numpy()

# Identify subclass kWh columns and the weighted-typical series
subclass_cols = [c for c in df.columns if c.lower().startswith("subclass_") and c.lower().endswith("_kwh")]
if not subclass_cols:
    raise ValueError("No 'Subclass_*_kWh' columns found.")
TYP_COL = "Weighted_kWh_avg_by_HousingStock"
if TYP_COL not in df.columns:
    raise ValueError(f"'{TYP_COL}' not found in enriched file.")

# Ensure numeric kWh
for c in subclass_cols + [TYP_COL]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

price_kwh = df["DAM_price_uah_per_kwh"].to_numpy(dtype=float)
typ_kwh   = df[TYP_COL].to_numpy(dtype=float)

# ---------------------------
# 2) Define ToU hour sets
# ---------------------------
# Two-zone
H_2Z_NIGHT = np.array([23] + list(range(0, 7)), dtype=int)  # 23,0..6
H_2Z_DAY   = np.array(list(range(7, 23)), dtype=int)        # 7..22

# Three-zone
H_3Z_OFFPEAK   = H_2Z_NIGHT.copy()                          # 23,0..6
H_3Z_ONPEAK    = np.array(list(range(8, 11)) + list(range(20, 22)), dtype=int)   # 8..10, 20..21
H_3Z_SEMIPEAK  = np.array([7] + list(range(11, 20)) + [22], dtype=int)          # 7, 11..19, 22

# ---------------------------
# 3) Per-subclass averages (overall + ToU zones)
# ---------------------------
rows = []
for col in subclass_cols:
    kwh = df[col].to_numpy(dtype=float)

    # Overall avg cost
    mask_price = ~np.isnan(price_kwh)
    q_all = kwh[mask_price]
    p_all = price_kwh[mask_price]
    total_q_all = float(np.sum(q_all))
    avg_all = float(np.sum(p_all * q_all) / total_q_all) if total_q_all > 0 else np.nan

    # ToU zone averages
    cost_2z_day   = _weighted_avg_uah_per_kwh(price_kwh, kwh, hour_of_day, H_2Z_DAY)
    cost_2z_night = _weighted_avg_uah_per_kwh(price_kwh, kwh, hour_of_day, H_2Z_NIGHT)

    cost_3z_off   = _weighted_avg_uah_per_kwh(price_kwh, kwh, hour_of_day, H_3Z_OFFPEAK)
    cost_3z_semi  = _weighted_avg_uah_per_kwh(price_kwh, kwh, hour_of_day, H_3Z_SEMIPEAK)
    cost_3z_on    = _weighted_avg_uah_per_kwh(price_kwh, kwh, hour_of_day, H_3Z_ONPEAK)

    rows.append({
        "sub_class_id": col.replace("_kWh", ""),
        "avg_cost_uah_per_kwh": avg_all,
        "cost_2zone_day":   cost_2z_day,
        "cost_2zone_night": cost_2z_night,
        "cost_3zone_off_peak":  cost_3z_off,
        "cost_3zone_semi_peak": cost_3z_semi,
        "cost_3zone_on_peak":   cost_3z_on,
        "total_kwh": total_q_all
    })

df_sub = pd.DataFrame(rows).sort_values("sub_class_id")

# ---------------------------
# 4) "Typical consumer" overall + ToU zone averages
# ---------------------------
avg_all_typ   = _weighted_avg_uah_per_kwh(price_kwh, typ_kwh, hour_of_day, np.arange(24, dtype=int))
cost_2z_day_T = _weighted_avg_uah_per_kwh(price_kwh, typ_kwh, hour_of_day, H_2Z_DAY)
cost_2z_nig_T = _weighted_avg_uah_per_kwh(price_kwh, typ_kwh, hour_of_day, H_2Z_NIGHT)

cost_3z_off_T  = _weighted_avg_uah_per_kwh(price_kwh, typ_kwh, hour_of_day, H_3Z_OFFPEAK)
cost_3z_semi_T = _weighted_avg_uah_per_kwh(price_kwh, typ_kwh, hour_of_day, H_3Z_SEMIPEAK)
cost_3z_on_T   = _weighted_avg_uah_per_kwh(price_kwh, typ_kwh, hour_of_day, H_3Z_ONPEAK)

# ---------------------------
# 5) DAM VWAP (overall) and ToU-VWAPs by zone
# ---------------------------
price_mwh = df["DAM_price_uah_per_mwh"].to_numpy(dtype=float)
vol_mwh   = df["DAM_volume_sell_mwh"].to_numpy(dtype=float)

# Overall
mask_vwap = ~np.isnan(price_mwh) & ~np.isnan(vol_mwh) & (vol_mwh > 0)
vwap_uah_per_mwh = float(np.sum(price_mwh[mask_vwap] * vol_mwh[mask_vwap]) / np.sum(vol_mwh[mask_vwap])) if np.sum(vol_mwh[mask_vwap]) > 0 else np.nan
vwap_uah_per_kwh = vwap_uah_per_mwh / 1000.0 if pd.notna(vwap_uah_per_mwh) else np.nan

# By zone (UAH/kWh directly)
tou_2z_day_from_vwap   = _vwap_uah_per_kwh_by_hours(price_mwh, vol_mwh, hour_of_day, H_2Z_DAY)
tou_2z_night_from_vwap = _vwap_uah_per_kwh_by_hours(price_mwh, vol_mwh, hour_of_day, H_2Z_NIGHT)

tou_3z_off_from_vwap   = _vwap_uah_per_kwh_by_hours(price_mwh, vol_mwh, hour_of_day, H_3Z_OFFPEAK)
tou_3z_semi_from_vwap  = _vwap_uah_per_kwh_by_hours(price_mwh, vol_mwh, hour_of_day, H_3Z_SEMIPEAK)
tou_3z_on_from_vwap    = _vwap_uah_per_kwh_by_hours(price_mwh, vol_mwh, hour_of_day, H_3Z_ONPEAK)

# ---------------------------
# 6) ToU tariffs from flat base (3.6 UAH/kWh)
# ---------------------------
tou_2z_day_from_flat   = 1.0 * FLAT_TARIFF_UAH_PER_KWH
tou_2z_night_from_flat = 0.5 * FLAT_TARIFF_UAH_PER_KWH

tou_3z_off_from_flat   = 0.4 * FLAT_TARIFF_UAH_PER_KWH
tou_3z_semi_from_flat  = 1.0 * FLAT_TARIFF_UAH_PER_KWH
tou_3z_on_from_flat    = 1.5 * FLAT_TARIFF_UAH_PER_KWH

# ---------------------------
# 7) Write outputs (no % differences) — robust engine selection
# ---------------------------
with pd.ExcelWriter(OUTPUT_PATH, engine=EXCEL_ENGINE) as writer:
    # Sheet 1: subclass averages + ToU
    df_sub.to_excel(writer, sheet_name="subclass_avg_cost", index=False)

    # Sheet 2: overall + ToU (typical consumer) + flat tariff + DAM VWAP + ToU tariffs (flat & VWAP-by-zone)
    metrics = [
        ("overall_weighted_avg_uah_per_kwh", avg_all_typ),

        ("cost_2zone_day_typical",   cost_2z_day_T),
        ("cost_2zone_night_typical", cost_2z_nig_T),

        ("cost_3zone_off_peak_typical",  cost_3z_off_T),
        ("cost_3zone_semi_peak_typical", cost_3z_semi_T),
        ("cost_3zone_on_peak_typical",   cost_3z_on_T),

        ("flat_tariff_uah_per_kwh", FLAT_TARIFF_UAH_PER_KWH),

        ("dam_vwap_uah_per_mwh", vwap_uah_per_mwh),
        ("dam_vwap_uah_per_kwh", vwap_uah_per_kwh),

        # ToU tariffs from flat base
        ("tou_2z_day_from_flat",   tou_2z_day_from_flat),
        ("tou_2z_night_from_flat", tou_2z_night_from_flat),
        ("tou_3z_off_from_flat",   tou_3z_off_from_flat),
        ("tou_3z_semi_from_flat",  tou_3z_semi_from_flat),
        ("tou_3z_on_from_flat",    tou_3z_on_from_flat),

        # ToU tariffs from DAM VWAP by zone (volume-weighted), in UAH/kWh
        ("tou_2z_day_from_vwap",   tou_2z_day_from_vwap),
        ("tou_2z_night_from_vwap", tou_2z_night_from_vwap),
        ("tou_3z_off_from_vwap",   tou_3z_off_from_vwap),
        ("tou_3z_semi_from_vwap",  tou_3z_semi_from_vwap),
        ("tou_3z_on_from_vwap",    tou_3z_on_from_vwap),
    ]
    pd.DataFrame(metrics, columns=["metric", "value"]).to_excel(writer, sheet_name="overall_and_tariff", index=False)

print("Done.")
print(f"Saved results to: {OUTPUT_PATH}")


[Cost Cell] Using Excel engine: openpyxl
Done.
Saved results to: C:\Users\admin\Bess_Valuation\subclass_costs_summary.xlsx


In [14]:
pip install matplotlib

     ---------------------------------------- 0.0/114.6 kB ? eta -:--:--
     --------------------------- ----------- 81.9/114.6 kB 2.3 MB/s eta 0:00:01
     -------------------------------------- 114.6/114.6 kB 2.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
    --------------------------------------- 0.2/8.1 MB 5.9 MB/s eta 0:00:02
   - -------------------------------------- 0.4/8.1 MB 6.3 MB/s eta 0:00:02
   --- ------------------------------------ 0.6/8.1 MB 4.8 MB/s eta 0:00:02
   ---- ----------------------------------- 0.8/8.1 MB 4.7 MB/s eta 0:00:02
   ----- ---------------------------------- 1.1/8.1 MB 4.9 MB/s eta 0:00:02
   ------ --------------------------------- 1.2/8.1 MB 4.9 MB/s eta 0:00:02
   ------- -------------------------------- 1.4/8.1 MB 4.8 MB/s eta 0:00:02
   -------- ------------------------------- 1.6/8.1 MB 4.5 MB/s eta 0:00:02
   -------- ------------------------------- 1.8/8.1 MB 4.6 MB/s eta 0:00:02
   --------- -


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\admin\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [16]:
# =========================
# Cell — 24h normalized histograms (subclasses, weighted, DAM) + single-page PDF
#   Source: the enriched file with aligned DAM columns
# =========================
# Inputs (edit these)
ENRICHED_PATH = r"C:\Users\admin\Bess_Valuation\Subclass_All_kWh__with_DAM_cols.xlsx"
PLOTS_DIR     = r"C:\Users\admin\Bess_Valuation\hist_png"
PDF_OUTPUT    = r"C:\Users\admin\Bess_Valuation\histograms_all_on_one_page.pdf"

# ---- No edits below this line ----
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

# ---------------------------
# Helpers
# ---------------------------
def _hour_shares(values: np.ndarray, hour_of_day: np.ndarray) -> np.ndarray:
    """
    Compute 24-element shares (%) by hour_of_day (0..23).
    Missing values are treated as 0 before aggregation.
    """
    v = np.nan_to_num(values.astype(float), nan=0.0)
    shares = np.zeros(24, dtype=float)
    for h in range(24):
        shares[h] = v[hour_of_day == h].sum()
    total = shares.sum()
    return (shares / total * 100.0) if total > 0 else np.full(24, np.nan)

def _plot_bar_24(shares_pct: np.ndarray, title: str, png_path: str):
    fig, ax = plt.subplots(figsize=(8, 3.0))
    ax.bar(range(24), shares_pct)
    ax.set_xticks(range(0, 24, 2))
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("Share of total (%)")
    ax.set_title(title)
    ymax = np.nanmax(shares_pct) if np.isfinite(np.nanmax(shares_pct)) else 5.0
    ax.set_ylim(0, max(5.0, ymax * 1.1))
    ax.grid(True, axis="y", linestyle=":", linewidth=0.5)
    fig.tight_layout()
    fig.savefig(png_path, dpi=150)
    plt.close(fig)

# ---------------------------
# 1) Read enriched file
# ---------------------------
df = pd.read_excel(ENRICHED_PATH)

# Compute hour-of-day from settlement_hour if present; else derive from timestamp
if "settlement_hour" in df.columns:
    # Rule: 00:00 ⇒ sh=1 … 23:00 ⇒ sh=24, so hour_of_day = sh-1
    hour_of_day = (pd.to_numeric(df["settlement_hour"], errors="coerce").fillna(1).astype(int) - 1).clip(0, 23).to_numpy()
else:
    if "timestamp" not in df.columns:
        raise ValueError("Neither 'settlement_hour' nor 'timestamp' found in the enriched file.")
    hour_of_day = pd.to_datetime(df["timestamp"]).dt.hour.to_numpy()

# Identify series
subclass_cols = [c for c in df.columns if c.lower().startswith("subclass_") and c.lower().endswith("_kwh")]
if not subclass_cols:
    raise ValueError("No 'Subclass_*_kWh' columns found in the enriched file.")

TYP_COL = "Weighted_kWh_avg_by_HousingStock"
if TYP_COL not in df.columns:
    raise ValueError(f"'{TYP_COL}' not found in the enriched file.")

DAM_COL = "DAM_volume_sell_mwh"
if DAM_COL not in df.columns:
    raise ValueError(f"'{DAM_COL}' not found in the enriched file.")

# ---------------------------
# 2) Build and save per-entity charts (normalized shares)
# ---------------------------
Path(PLOTS_DIR).mkdir(parents=True, exist_ok=True)
png_paths = []

# Subclasses
for col in subclass_cols:
    shares = _hour_shares(pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=float), hour_of_day)
    fname = f"hist_{col.replace('_kWh','')}.png"
    png = os.path.join(PLOTS_DIR, fname)
    _plot_bar_24(shares, title=col.replace('_kWh',''), png_path=png)
    png_paths.append(png)

# Weighted "typical consumer"
shares_weighted = _hour_shares(pd.to_numeric(df[TYP_COL], errors="coerce").to_numpy(dtype=float), hour_of_day)
png_weighted = os.path.join(PLOTS_DIR, "hist_Weighted.png")
_plot_bar_24(shares_weighted, title="Weighted (HousingStock)", png_path=png_weighted)
png_paths.append(png_weighted)

# DAM volume (as % of total)
shares_dam = _hour_shares(pd.to_numeric(df[DAM_COL], errors="coerce").to_numpy(dtype=float), hour_of_day)
png_dam = os.path.join(PLOTS_DIR, "hist_DAM.png")
_plot_bar_24(shares_dam, title="DAM volume", png_path=png_dam)
png_paths.append(png_dam)

print(f"Saved {len(png_paths)} PNG charts to: {PLOTS_DIR}")

# ---------------------------
# 3) Single-page PDF with all PNGs in a grid
# ---------------------------
n = len(png_paths)
ncols = 5
nrows = math.ceil(n / ncols)
fig_w = 3.2 * ncols
fig_h = 2.6 * nrows

fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h))
axes = np.array(axes).reshape(nrows, ncols)

for i, ax in enumerate(axes.ravel()):
    ax.axis("off")
    if i < n:
        img = mpimg.imread(png_paths[i])
        ax.imshow(img)
        ax.set_title(Path(png_paths[i]).stem, fontsize=8, pad=2)

fig.tight_layout(pad=0.5)
fig.savefig(PDF_OUTPUT, dpi=200)
plt.close(fig)

print(f"Single-page PDF saved to: {PDF_OUTPUT}")


Saved 21 PNG charts to: C:\Users\admin\Bess_Valuation\hist_png
Single-page PDF saved to: C:\Users\admin\Bess_Valuation\histograms_all_on_one_page.pdf


In [2]:
# =========================
# Cell — 19 charts: Weighted avg DAM price by hour (last 12 months)
#   Source: Subclass_All_kWh__with_DAM_cols.xlsx
#   Output: 19 PNGs, 1 CSV (wide), 1 optional single-page PDF if matplotlib is installed
# =========================

# ---- Inputs (edit these) ----
ENRICHED_PATH = r"C:\Users\admin\Bess_Valuation\Subclass_All_kWh__with_DAM_cols.xlsx"
OUT_DIR       = r"C:\Users\admin\Bess_Valuation\dam_hourly_weighted_price"
PNG_DIR       = OUT_DIR + r"\png"
CSV_PATH      = OUT_DIR + r"\weighted_dam_price_by_hour.csv"
PDF_PATH      = OUT_DIR + r"\weighted_dam_price_by_hour_ALL.pdf"

# ---- No edits below this line ----
import os
import math
import numpy as np
import pandas as pd
from pathlib import Path

# Optional plotting (robust if matplotlib missing)
HAVE_MPL = True
try:
    import matplotlib.pyplot as plt
    import matplotlib.image as mpimg
except Exception:
    HAVE_MPL = False

# ---------- Helpers ----------
def _to_num(x):
    return pd.to_numeric(pd.Series(x, dtype="string").str.replace(",", "", regex=False), errors="coerce")

def _weighted_avg_by_hour(price_mwh: np.ndarray,
                          weights: np.ndarray,
                          hour_of_day: np.ndarray) -> np.ndarray:
    """
    Returns a 24-length array of weighted avg price (UAH/MWh) per hour 0..23:
    sum(price * weight) / sum(weight) over rows of that hour.
    If sum(weight)==0 for an hour → NaN.
    """
    p = price_mwh.astype(float)
    w = np.nan_to_num(weights.astype(float), nan=0.0)
    h = hour_of_day.astype(int)

    out = np.full(24, np.nan, dtype=float)
    for hr in range(24):
        m = (h == hr) & ~np.isnan(p) & (w >= 0)  # allow zero; it just won't contribute
        denom = w[m].sum()
        if denom > 0:
            out[hr] = (p[m] * w[m]).sum() / denom
        else:
            out[hr] = np.nan
    return out

def _save_bar_24(values: np.ndarray, title: str, png_path: str):
    fig, ax = plt.subplots(figsize=(8, 3.0))
    ax.bar(range(24), values)
    ax.set_xticks(range(0, 24, 2))
    ax.set_xlabel("Hour of day (0–23)")
    ax.set_ylabel("Weighted avg price (UAH/MWh)")
    ax.set_title(title)
    ymax = np.nanmax(values) if np.isfinite(np.nanmax(values)) else 1000.0
    ax.set_ylim(0, max(ymax * 1.1, 10.0))
    ax.grid(True, axis="y", linestyle=":", linewidth=0.6)
    fig.tight_layout()
    fig.savefig(png_path, dpi=150)
    plt.close(fig)

# ---------- Load data ----------
df = pd.read_excel(ENRICHED_PATH)

# Parse timestamp if present; use last 12 months (8,760 rows).
if "timestamp" in df.columns:
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.assign(_ts=ts).sort_values("_ts").reset_index(drop=True)
    # If we have >= 8760 rows, take the last 8760; else take all.
    if len(df) >= 8760:
        df = df.iloc[-8760:].copy()
else:
    # If no timestamp, assume the file is already trimmed to last 12 months
    pass

# Hour of day from settlement_hour if present; else from timestamp (fallback)
if "settlement_hour" in df.columns:
    hour_of_day = (pd.to_numeric(df["settlement_hour"], errors="coerce").fillna(1).astype(int) - 1).clip(0, 23).to_numpy()
else:
    if "timestamp" not in df.columns:
        raise ValueError("Neither 'settlement_hour' nor 'timestamp' found; cannot compute hour-of-day.")
    hour_of_day = pd.to_datetime(df["timestamp"]).dt.hour.to_numpy()

# Clean price array (UAH/MWh)
if "DAM_price_uah_per_mwh" not in df.columns:
    raise ValueError("Column 'DAM_price_uah_per_mwh' not found.")
price_mwh = _to_num(df["DAM_price_uah_per_mwh"]).to_numpy(dtype=float)

# Collect subclass weight columns
subclass_cols = [c for c in df.columns if c.lower().startswith("subclass_") and c.lower().endswith("_kwh")]
if not subclass_cols:
    raise ValueError("No 'Subclass_*_kWh' columns found.")

# Sort lexicographically so Subclass_01..Subclass_19 are in order, then pick up to 19
subclass_cols = sorted(subclass_cols)
entities = [(c.replace("_kWh", ""), df[c].to_numpy(dtype=float)) for c in subclass_cols[:19]]

# If we have fewer than 19 subclass columns, add "Weighted" to fill (optional)
if len(entities) < 19 and "Weighted_kWh_avg_by_HousingStock" in df.columns:
    entities.append(("Weighted", pd.to_numeric(df["Weighted_kWh_avg_by_HousingStock"], errors="coerce").fillna(0.0).to_numpy(dtype=float)))

# Ensure exactly 19 if possible (if more subclass cols exist we already capped at 19)
entities = entities[:19]

# ---------- Compute weighted avg prices by hour for each entity ----------
rows_wide = []
charts = []  # (name, values)

for name, w_arr in entities:
    vals = _weighted_avg_by_hour(price_mwh=price_mwh, weights=w_arr, hour_of_day=hour_of_day)
    # wide row
    row = {"entity": name}
    for hr in range(24):
        row[f"h{hr:02d}"] = vals[hr]
    rows_wide.append(row)
    charts.append((name, vals))

# ---------- Save outputs ----------
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(PNG_DIR).mkdir(parents=True, exist_ok=True)

# CSV (wide)
wide_df = pd.DataFrame(rows_wide).sort_values("entity")
wide_df.to_csv(CSV_PATH, index=False)
print(f"[OK] CSV saved: {CSV_PATH}")

# PNGs + PDF (if matplotlib available)
if HAVE_MPL:
    png_paths = []
    for name, vals in charts:
        png = os.path.join(PNG_DIR, f"dam_weighted_price_{name}.png")
        _save_bar_24(vals, title=f"{name} — Weighted avg DAM price by hour", png_path=png)
        png_paths.append(png)
    print(f"[OK] {len(png_paths)} PNG charts saved to: {PNG_DIR}")

    # Single-page PDF (grid)
    n = len(png_paths)
    ncols = 5
    nrows = math.ceil(n / ncols)
    fig_w = 3.2 * ncols
    fig_h = 2.6 * nrows

    import matplotlib.pyplot as plt  # already imported; safe
    import matplotlib.image as mpimg

    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h))
    axes = np.array(axes).reshape(nrows, ncols)

    for i, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if i < n:
            img = mpimg.imread(png_paths[i])
            ax.imshow(img)
            ax.set_title(Path(png_paths[i]).stem, fontsize=8, pad=2)

    fig.tight_layout(pad=0.5)
    fig.savefig(PDF_PATH, dpi=200)
    plt.close(fig)
    print(f"[OK] Single-page PDF saved: {PDF_PATH}")
else:
    print(
        "matplotlib not installed — skipped PNG/PDF generation.\n"
        "Install with:  pip install matplotlib\n"
        f"You still have the CSV with all weighted hourly prices: {CSV_PATH}"
    )


[OK] CSV saved: C:\Users\admin\Bess_Valuation\dam_hourly_weighted_price\weighted_dam_price_by_hour.csv
[OK] 19 PNG charts saved to: C:\Users\admin\Bess_Valuation\dam_hourly_weighted_price\png
[OK] Single-page PDF saved: C:\Users\admin\Bess_Valuation\dam_hourly_weighted_price\weighted_dam_price_by_hour_ALL.pdf


In [8]:
# =========================
# Cell — 19 combined charts per subclass:
#   Box plots (kWh) + green line (weighted avg DAM price, UAH/MWh)
#   Axes policy:
#     - Left y-axis (kWh) ALWAYS starts at 0
#     - Right y-axis (UAH/MWh) starts at 0, UNLESS any price < 0 → starts at that negative min
#   Uses last 12 months only (last 8,760 rows if timestamp is present)
#   Robust to missing matplotlib (CSV-only fallback)
# =========================

# --- Inputs (edit these paths) ---
ENRICHED_PATH = r"C:\Users\admin\Bess_Valuation\Subclass_All_kWh__with_DAM_cols.xlsx"
OUT_DIR       = r"C:\Users\admin\Bess_Valuation\combined_boxline"
PNG_DIR       = OUT_DIR + r"\png"
CSV_BOX       = OUT_DIR + r"\box_stats_by_hour.csv"          # long format
CSV_PRICE     = OUT_DIR + r"\weighted_dam_price_by_hour.csv" # wide format
PDF_PATH      = OUT_DIR + r"\combined_boxline_ALL.pdf"

# --- No edits below ---
import os
import math
import numpy as np
import pandas as pd
from pathlib import Path

# Optional plotting (robust if matplotlib missing)
HAVE_MPL = True
try:
    import matplotlib.pyplot as plt
    import matplotlib.image as mpimg
except Exception:
    HAVE_MPL = False

# ---------- Helpers ----------
def _to_num(x):
    """Coerce to numeric; tolerate commas; return NaN on failure."""
    return pd.to_numeric(pd.Series(x, dtype="string").str.replace(",", "", regex=False), errors="coerce")

def _hour_of_day_from_df(df: pd.DataFrame) -> np.ndarray:
    """
    Returns hour_of_day (0..23). Prefer settlement_hour (1..24): hour = sh-1.
    Fallback to parsing from timestamp.
    """
    if "settlement_hour" in df.columns:
        return (pd.to_numeric(df["settlement_hour"], errors="coerce").fillna(1).astype(int) - 1).clip(0, 23).to_numpy()
    if "timestamp" in df.columns:
        return pd.to_datetime(df["timestamp"], errors="coerce").dt.hour.to_numpy()
    raise ValueError("Neither 'settlement_hour' nor 'timestamp' found; cannot compute hour-of-day.")

def _last_year_slice(df: pd.DataFrame) -> pd.DataFrame:
    """
    If timestamp exists, sort by timestamp and take last 8,760 rows.
    Otherwise, return as-is (assumed already trimmed).
    """
    if "timestamp" in df.columns:
        ts = pd.to_datetime(df["timestamp"], errors="coerce")
        out = df.assign(_ts=ts).sort_values("_ts").reset_index(drop=True)
        if len(out) >= 8760:
            out = out.iloc[-8760:].copy()
        out.drop(columns=["_ts"], inplace=True, errors="ignore")
        return out
    return df

def _box_stats_by_hour(values: np.ndarray, hour_of_day: np.ndarray) -> pd.DataFrame:
    """
    Compute boxplot stats per hour 0..23 for 'values' (kWh).
    Returns long DataFrame: hour, min, q1, median, q3, max, count
    """
    vals = values.astype(float)
    rows = []
    for hr in range(24):
        m = (hour_of_day == hr)
        data = vals[m]
        data = data[~np.isnan(data)]
        if data.size == 0:
            rows.append({
                "hour": hr, "min": np.nan, "q1": np.nan, "median": np.nan,
                "q3": np.nan, "max": np.nan, "count": 0
            })
            continue
        q1 = np.percentile(data, 25)
        med = np.percentile(data, 50)
        q3 = np.percentile(data, 75)
        rows.append({
            "hour": hr,
            "min": float(np.min(data)),
            "q1": float(q1),
            "median": float(med),
            "q3": float(q3),
            "max": float(np.max(data)),
            "count": int(data.size),
        })
    return pd.DataFrame(rows)

def _weighted_avg_price_by_hour(price_mwh: np.ndarray, weights: np.ndarray, hour_of_day: np.ndarray) -> np.ndarray:
    """
    Weighted average price (UAH/MWh) by hour 0..23:
    sum(price * weight) / sum(weight). If sum(weight)==0 -> NaN.
    """
    p = price_mwh.astype(float)
    w = np.nan_to_num(weights.astype(float), nan=0.0)
    h = hour_of_day.astype(int)
    out = np.full(24, np.nan, dtype=float)
    for hr in range(24):
        m = (h == hr) & ~np.isnan(p) & (w >= 0)
        denom = w[m].sum()
        out[hr] = (p[m] * w[m]).sum() / denom if denom > 0 else np.nan
    return out

def _plot_box_with_line(hours_data_for_box,  # list of arrays (one per hour) for boxplot
                        price_line_24,        # 24-length array (UAH/MWh)
                        title, png_path):
    """
    Draws a combined chart:
      - box plots (kWh) on left y-axis   → LEFT axis ALWAYS starts at 0
      - weighted avg price (UAH/MWh)     → RIGHT axis starts at 0, unless any price < 0 → starts at that negative min
    """
    fig, ax = plt.subplots(figsize=(10, 4))

    # --- Box plots (left axis: kWh) ---
    bp = ax.boxplot(hours_data_for_box, positions=list(range(24)), showfliers=False, widths=0.6)
    ax.set_xlabel("Hour of day (0–23)")
    ax.set_ylabel("Consumption (kWh)")
    ax.set_xticks(range(0, 24, 2))
    ax.grid(True, axis="y", linestyle=":", linewidth=0.6)
    ax.set_title(title)

    # Force LEFT axis to start at 0, compute a reasonable top:
    all_vals = [arr[~np.isnan(arr)] for arr in hours_data_for_box if arr.size > 0]
    if len(all_vals) > 0 and np.concatenate(all_vals).size > 0:
        max_kwh = float(np.nanmax(np.concatenate(all_vals)))
        top_left = max_kwh * 1.1 if max_kwh > 0 else 1.0
    else:
        top_left = 1.0
    ax.set_ylim(0.0, top_left)

    # --- Price line (right axis: UAH/MWh) ---
    ax2 = ax.twinx()
    ax2.plot(range(24), price_line_24, marker="o", linewidth=1.8, color="green", label="Weighted avg DAM price (UAH/MWh)")
    ax2.set_ylabel("Price (UAH/MWh)")

    # RIGHT axis policy: start from 0 unless there's any negative value
    finite_prices = np.array(price_line_24, dtype=float)
    finite_prices = finite_prices[np.isfinite(finite_prices)]
    if finite_prices.size > 0:
        pmin = float(np.min(finite_prices))
        pmax = float(np.max(finite_prices))
        bottom_right = pmin if pmin < 0 else 0.0
        # Pad the top a bit; ensure it's above bottom
        span = max(pmax - pmin, 1e-6)
        top_right = pmax + 0.1 * span
        if top_right <= bottom_right:
            top_right = bottom_right + max(abs(bottom_right) * 0.2, 1.0)
        ax2.set_ylim(bottom_right, top_right)
    else:
        # No data → 0..1 as a safe default
        ax2.set_ylim(0.0, 1.0)

    # Legend (box + line)
    from matplotlib.lines import Line2D
    box_proxy = Line2D([0], [0], linestyle="none", marker="s", color="black", label="Consumption (box plots)")
    line_proxy = Line2D([0], [0], color="green", label="Weighted avg DAM price (UAH/MWh)")
    ax.legend(handles=[box_proxy, line_proxy], loc="upper left")

    fig.tight_layout()
    fig.savefig(png_path, dpi=150)
    plt.close(fig)

# ---------- Load data ----------
df = pd.read_excel(ENRICHED_PATH)

# Trim to last 12 months
df = _last_year_slice(df)

# Hour-of-day
hour_of_day = _hour_of_day_from_df(df)

# Price (UAH/MWh)
if "DAM_price_uah_per_mwh" not in df.columns:
    raise ValueError("Column 'DAM_price_uah_per_mwh' not found.")
price_mwh = _to_num(df["DAM_price_uah_per_mwh"]).to_numpy(dtype=float)

# Subclass columns
subclass_cols = sorted([c for c in df.columns if c.lower().startswith("subclass_") and c.lower().endswith("_kwh")])
if len(subclass_cols) < 19:
    raise ValueError(f"Expected at least 19 subclass kWh columns, found {len(subclass_cols)}.")
subclass_cols = subclass_cols[:19]  # exactly 19

# Prepare output dirs
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(PNG_DIR).mkdir(parents=True, exist_ok=True)

# Accumulators for CSVs
box_stats_all = []   # list of DataFrames with entity column
price_rows = []      # wide rows: entity, h00..h23

# ---- Compute & (optionally) plot for each subclass ----
png_paths = []

for col in subclass_cols:
    entity = col.replace("_kWh", "")
    kwh = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=float)

    # Box stats (per hour)
    stats_df = _box_stats_by_hour(kwh, hour_of_day)
    stats_df.insert(0, "entity", entity)
    box_stats_all.append(stats_df)

    # For plotting boxplots: raw samples per hour
    hours_data_for_box = []
    for hr in range(24):
        sample = kwh[hour_of_day == hr]
        sample = sample[~np.isnan(sample)]
        hours_data_for_box.append(sample)

    # Weighted average price per hour (UAH/MWh)
    price_by_hour = _weighted_avg_price_by_hour(price_mwh, kwh, hour_of_day)

    # Add to price CSV (wide)
    row = {"entity": entity}
    for hr in range(24):
        row[f"h{hr:02d}"] = price_by_hour[hr]
    price_rows.append(row)

    # Plot PNG (if matplotlib installed)
    if HAVE_MPL:
        png_path = os.path.join(PNG_DIR, f"combined_boxline_{entity}.png")
        _plot_box_with_line(hours_data_for_box, price_by_hour, title=entity, png_path=png_path)
        png_paths.append(png_path)

# ---- Save CSVs ----
box_stats_df = pd.concat(box_stats_all, ignore_index=True)
box_stats_df.to_csv(CSV_BOX, index=False)
price_wide_df = pd.DataFrame(price_rows).sort_values("entity")
price_wide_df.to_csv(CSV_PRICE, index=False)

print(f"[OK] Box stats CSV: {CSV_BOX}")
print(f"[OK] Hourly weighted price CSV: {CSV_PRICE}")

# ---- Single-page PDF (grid of all charts) if matplotlib present ----
if HAVE_MPL and len(png_paths) == 19:
    n = len(png_paths)
    ncols = 5
    nrows = math.ceil(n / ncols)
    fig_w = 3.3 * ncols
    fig_h = 2.6 * nrows

    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h))
    axes = np.array(axes).reshape(nrows, ncols)

    for i, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if i < n:
            img = mpimg.imread(png_paths[i])
            ax.imshow(img)
            ax.set_title(Path(png_paths[i]).stem, fontsize=8, pad=2)

    fig.tight_layout(pad=0.5)
    fig.savefig(PDF_PATH, dpi=200)
    plt.close(fig)
    print(f"[OK] Single-page PDF: {PDF_PATH}")
elif not HAVE_MPL:
    print("matplotlib not installed — skipped PNG/PDF generation. Install with:  pip install matplotlib")
else:
    print(f"PNG count != 19 (got {len(png_paths)}); PDF not created.")


[OK] Box stats CSV: C:\Users\admin\Bess_Valuation\combined_boxline\box_stats_by_hour.csv
[OK] Hourly weighted price CSV: C:\Users\admin\Bess_Valuation\combined_boxline\weighted_dam_price_by_hour.csv
[OK] Single-page PDF: C:\Users\admin\Bess_Valuation\combined_boxline\combined_boxline_ALL.pdf


In [10]:
%pip install reportlab


   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
    --------------------------------------- 0.0/2.0 MB 660.6 kB/s eta 0:00:03
   --- ------------------------------------ 0.2/2.0 MB 2.1 MB/s eta 0:00:01
   -------- ------------------------------- 0.4/2.0 MB 3.3 MB/s eta 0:00:01
   --------------- ------------------------ 0.7/2.0 MB 4.2 MB/s eta 0:00:01
   --------------------- ------------------ 1.0/2.0 MB 4.7 MB/s eta 0:00:01
   ---------------------------- ----------- 1.4/2.0 MB 5.2 MB/s eta 0:00:01
   ------------------------------------ --- 1.8/2.0 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 5.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\admin\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [4]:
%pip install xlsxwriter openpyxl


   ---------------------------------------- 0.0/175.3 kB ? eta -:--:--
   -- ------------------------------------- 10.2/175.3 kB ? eta -:--:--
   ------ -------------------------------- 30.7/175.3 kB 435.7 kB/s eta 0:00:01
   --------- ----------------------------- 41.0/175.3 kB 393.8 kB/s eta 0:00:01
   ------------------ -------------------- 81.9/175.3 kB 651.6 kB/s eta 0:00:01
   ------------------------------- ------ 143.4/175.3 kB 847.9 kB/s eta 0:00:01
   -------------------------------------- 175.3/175.3 kB 878.1 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\admin\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [6]:
# =========================
# Cell — Distribution stats + box plots for bldg_id CSV (ALL-YEAR and BY-ZONE)
# Also adds RMSD sheet for 3Z_ON morning/evening per series (each bldg_id column)
# =========================
import os, math
import numpy as np
import pandas as pd
from pathlib import Path

# plotting / PDF
import matplotlib
import matplotlib.pyplot as plt
from reportlab.lib.pagesizes import A1
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader

# ---------- Inputs (edit these) ----------
CSV_PATH   = r"C:\Users\admin\load_profiles_2024_r2\subclass_reports\bldg_hourly_filtered_by_weights_gt0.csv"
OUTPUT_DIR = Path(r"C:\Users\admin\load_profiles_2024_r2\subclass_reports\distribution_outputs_bldg")
INCLUDE_STATS_ON_CHART = True   # annotate mean/std/cv on each box

# Zones (hour_of_day in [0..23])
H_2Z_DAY         = np.array(list(range(7, 23)), dtype=int)           # 07:00–22:59
H_3Z_ON_MORNING  = np.array(list(range(8, 11)), dtype=int)           # 08:00–10:59
H_3Z_ON_EVENING  = np.array(list(range(20, 22)), dtype=int)          # 20:00–21:59
H_3Z_ON          = np.concatenate([H_3Z_ON_MORNING, H_3Z_ON_EVENING])

# Outputs
EXCEL_ALL_YEAR   = OUTPUT_DIR / "bldg_all_year_percentiles.xlsx"
EXCEL_BY_ZONES   = OUTPUT_DIR / "bldg_timezones_statistics.xlsx"
PDF_BOX_ALL_YEAR = OUTPUT_DIR / "boxplots_all_series_onepage.pdf"
PDF_BOX_ZONES    = OUTPUT_DIR / "boxplots_by_zone_grid_onepage.pdf"
PNG_DIR_ALL      = OUTPUT_DIR / "png_all_year"
PNG_DIR_ZONES    = OUTPUT_DIR / "png_by_zone"

# ---------- Helpers ----------
def ensure_dirs():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PNG_DIR_ALL.mkdir(parents=True, exist_ok=True)
    PNG_DIR_ZONES.mkdir(parents=True, exist_ok=True)

def hour_of_day(df: pd.DataFrame) -> np.ndarray:
    """
    Priority:
      1) if 'settlement_hour' exists → hod = settlement_hour-1 (clipped 0..23)
      2) else from 'hour_of_day' if present
      3) else parse from 'timestamp' (datetime hour)
    """
    if "settlement_hour" in df.columns:
        hod = (pd.to_numeric(df["settlement_hour"], errors="coerce").fillna(1).astype(int) - 1).clip(0, 23)
        return hod.to_numpy()
    if "hour_of_day" in df.columns:
        return pd.to_numeric(df["hour_of_day"], errors="coerce").fillna(0).astype(int).clip(0, 23).to_numpy()
    if "timestamp" in df.columns:
        ts = pd.to_datetime(df["timestamp"], errors="coerce")
        return ts.dt.hour.fillna(0).astype(int).to_numpy()
    raise ValueError("Need one of columns: 'settlement_hour', 'hour_of_day', or 'timestamp'.")

def identify_series_cols(df: pd.DataFrame):
    """
    Use all numeric columns except common time columns as series (bldg_id columns).
    Also include columns that LOOK like bldg_id (prefix) even if dtype is object, converting to numeric later.
    """
    exclude = {"timestamp", "DAM_Date", "settlement_hour", "hour_of_day"}
    numeric_cols = [c for c in df.columns if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]
    # allow non-numeric bldg columns (rare) — we will coerce to numeric later
    heuristic_cols = [c for c in df.columns if c not in exclude and str(c).lower().startswith(("bldg", "bldg_id", "id_"))]
    cols = sorted(set(numeric_cols).union(heuristic_cols))
    if not cols:
        raise ValueError("No series columns found (expected bldg_id* numeric columns).")
    return cols

def compute_percentiles(arr: np.ndarray, percentiles=(0,5,10,20,25,50,75,80,90,95,100)):
    x = np.nan_to_num(arr.astype(float), nan=0.0)
    prc_vals = np.percentile(x, percentiles)
    return dict(zip([f"p{p}" for p in percentiles], prc_vals))

def compute_stats(arr: np.ndarray):
    x = np.nan_to_num(arr.astype(float), nan=0.0)
    mean = float(np.mean(x))
    std  = float(np.std(x, ddof=0))
    cv   = float(std / mean) if mean > 0 else np.nan
    return {
        "mean": mean, "std": std, "cv": cv,
        "min": float(np.min(x)), "max": float(np.max(x)),
        "p10": float(np.percentile(x, 10)),
        "p50": float(np.percentile(x, 50)),
        "p90": float(np.percentile(x, 90)),
        "p25": float(np.percentile(x, 25)),
        "p75": float(np.percentile(x, 75)),
    }

def make_boxplot_png(series_name: str, data: np.ndarray, out_png: Path, include_stats: bool):
    # Custom box: whiskers = min/max; box = P10..P90; median line = P50
    x = np.nan_to_num(data.astype(float), nan=0.0)
    st = compute_stats(x)
    bxp_stats = {
        "med":    st["p50"],
        "q1":     st["p10"],
        "q3":     st["p90"],
        "whislo": st["min"],
        "whishi": st["max"],
        "fliers": []
    }
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.bxp([bxp_stats], showfliers=False)
    ax.set_title(series_name)
    ax.set_xticks([])
    ax.set_ylabel("kWh")
    if include_stats:
        txt = f"mean={st['mean']:.3f}\nstd={st['std']:.3f}\ncv={st['cv']:.3f}"
        ax.text(0.98, 0.98, txt, transform=ax.transAxes, ha="right", va="top", fontsize=8,
                bbox=dict(boxstyle="round", alpha=0.15))
    fig.tight_layout()
    fig.savefig(str(out_png), dpi=150)
    plt.close(fig)

def assemble_single_page_pdf(image_paths, out_pdf_path: Path, page_size=A1, margin=36):
    # Simple tiling
    c = canvas.Canvas(str(out_pdf_path), pagesize=page_size)
    page_w, page_h = page_size
    n = len(image_paths)
    if n == 0:
        c.drawString(100, page_h/2, "No images.")
        c.showPage(); c.save(); return
    cols = 5
    rows = math.ceil(n / cols)
    cell_w = (page_w - 2*margin) / cols
    cell_h = (page_h - 2*margin) / rows
    for idx, p in enumerate(image_paths):
        col = idx % cols
        row = idx // cols
        x = margin + col * cell_w
        y = page_h - margin - (row + 1) * cell_h
        try:
            img = ImageReader(str(p))
            iw, ih = img.getSize()
            s = min(cell_w/iw, cell_h/ih)
            c.drawImage(img, x + (cell_w - iw*s)/2, y + (cell_h - ih*s)/2, iw*s, ih*s)
        except:
            c.drawString(x+5, y+cell_h/2, f"Error: {Path(p).name}")
    c.showPage(); c.save()

def assemble_one_page_pdf_grid_by_series_zone(png_map, series_order, zone_order, out_pdf_path: Path, page_size=A1, margin=36):
    """rows = series_order, cols = zone_order; png_map[(series, zone)] -> image path"""
    c = canvas.Canvas(str(out_pdf_path), pagesize=page_size)
    page_w, page_h = page_size
    rows = len(series_order); cols = len(zone_order)
    if rows == 0 or cols == 0:
        c.drawString(100, page_h/2, "No images.")
        c.showPage(); c.save(); return

    cell_w = (page_w - 2*margin) / cols
    cell_h = (page_h - 2*margin) / rows

    # headers
    c.setFont("Helvetica-Bold", 10)
    for j, zone in enumerate(zone_order):
        x = margin + j * cell_w
        y = page_h - margin + 6
        c.drawString(x + 4, y, zone)

    # grid
    for i, series in enumerate(series_order):
        c.drawString(4, page_h - margin - i * cell_h - 12, series)
        for j, zone in enumerate(zone_order):
            x = margin + j * cell_w
            y = page_h - margin - (i + 1) * cell_h
            pth = png_map.get((series, zone))
            if pth and os.path.exists(pth):
                try:
                    img = ImageReader(str(pth))
                    iw, ih = img.getSize()
                    s = min(cell_w/iw, cell_h/ih)
                    c.drawImage(img, x + (cell_w - iw*s)/2, y + (cell_h - ih*s)/2, iw*s, ih*s)
                except:
                    c.drawString(x+5, y+cell_h/2, f"Error: {Path(pth).name}")
            else:
                c.rect(x+2, y+2, cell_w-4, cell_h-4)
                c.drawString(x+6, y+cell_h/2, "n/a")
    c.showPage(); c.save()

# ---------- Main ----------
ensure_dirs()

if not os.path.exists(CSV_PATH):
    print(f"Source file not found: {CSV_PATH}")
else:
    df = pd.read_csv(CSV_PATH, low_memory=False)

    # series columns (bldg_id*) and order
    series_cols = identify_series_cols(df)
    ordered_series = sorted(series_cols)

    # hour-of-day and day column
    hod = hour_of_day(df)
    if "DAM_Date" in df.columns:
        day_col = pd.to_datetime(df["DAM_Date"], errors="coerce").dt.normalize()
    elif "timestamp" in df.columns:
        day_col = pd.to_datetime(df["timestamp"], errors="coerce").dt.normalize()
    else:
        # fallback: build day from hour_of_day position assuming continuous hourly data
        day_col = pd.Series(pd.to_datetime("1970-01-01") + pd.to_timedelta(np.arange(len(df)), unit="h")).dt.normalize()

    # 1) ALL-YEAR percentiles + stats → Excel
    rows = []
    for col in ordered_series:
        x = pd.to_numeric(df[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        prc = compute_percentiles(x, percentiles=(0,5,10,20,25,50,75,80,90,95,100))
        st  = compute_stats(x)
        rows.append({"series": col, **prc, "mean": st["mean"], "std": st["std"], "cv": st["cv"]})
    df_all_year = pd.DataFrame(rows).sort_values("series")
    with pd.ExcelWriter(EXCEL_ALL_YEAR, engine="xlsxwriter") as w:
        df_all_year.to_excel(w, sheet_name="all_year_percentiles", index=False)

    # 2) ALL-YEAR boxplots → one-page PDF
    png_paths_all = []
    for col in ordered_series:
        x = pd.to_numeric(df[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        out_png = PNG_DIR_ALL / f"box_{col}.png"
        make_boxplot_png(col, x, out_png, INCLUDE_STATS_ON_CHART)
        png_paths_all.append(out_png)
    assemble_single_page_pdf([str(p) for p in png_paths_all], PDF_BOX_ALL_YEAR, page_size=A1)

    # 3) ZONE stats + zone boxplots (rows=series, cols=fixed order)
    ZONE_ORDER = ["2Z_DAY", "3Z_ON_morning", "3Z_ON_evening", "3Z_ON"]
    ZONES = {"2Z_DAY": H_2Z_DAY, "3Z_ON_morning": H_3Z_ON_MORNING, "3Z_ON_evening": H_3Z_ON_EVENING, "3Z_ON": H_3Z_ON}

    zone_tables = {}
    png_map = {}

    for zname, zhours in ZONES.items():
        rows_zone = []
        zmask = np.isin(hod, zhours)
        for col in ordered_series:
            x_full = pd.to_numeric(df[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
            x = x_full[zmask]
            prc = compute_percentiles(x, percentiles=(0,5,10,25,50,75,90,95,100))
            st  = compute_stats(x)
            rows_zone.append({"series": col, **prc, "mean": st["mean"], "std": st["std"], "cv": st["cv"]})
            out_png = PNG_DIR_ZONES / f"box_{col}__{zname}.png"
            make_boxplot_png(f"{col} | {zname}", x, out_png, INCLUDE_STATS_ON_CHART)
            png_map[(col, zname)] = str(out_png)

        zone_tables[zname] = pd.DataFrame(rows_zone).sort_values("series")

    # Save per-zone stats (each zone a separate sheet)
    with pd.ExcelWriter(EXCEL_BY_ZONES, engine="xlsxwriter") as w:
        for zname, d in zone_tables.items():
            d.to_excel(w, sheet_name=zname, index=False)

    # 4) One-page BY-ZONE PDF (grid)
    assemble_one_page_pdf_grid_by_series_zone(
        png_map=png_map,
        series_order=ordered_series,
        zone_order=ZONE_ORDER,
        out_pdf_path=PDF_BOX_ZONES,
        page_size=A1
    )

    # 5) EXTRA: RMSD across 3Z_ON morning/evening per series (daily sums)
    mask_morn = np.isin(hod, H_3Z_ON_MORNING)
    mask_even = np.isin(hod, H_3Z_ON_EVENING)

    rows_rmsd = []
    for col in ordered_series:
        x_all = pd.to_numeric(df[col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        # daily sums in morning/evening windows
        morn_daily = pd.DataFrame({"day": day_col[mask_morn], "val": x_all[mask_morn]}).groupby("day", as_index=False)["val"].sum()
        even_daily = pd.DataFrame({"day": day_col[mask_even], "val": x_all[mask_even]}).groupby("day", as_index=False)["val"].sum()
        daily = pd.merge(morn_daily, even_daily, on="day", how="outer", suffixes=("_morn", "_even")).fillna(0.0)

        N = int(len(daily))
        if N == 0:
            rows_rmsd.append({"series": col, "N_days": 0, "mean_morning": np.nan, "mean_evening": np.nan,
                              "c_star": np.nan, "RMSD": np.nan, "RMSD_over_c_star": np.nan})
            continue

        mean_m = float(daily["val_morn"].mean())
        mean_e = float(daily["val_even"].mean())
        c_star = 0.5 * (mean_m + mean_e)
        diffsq = (daily["val_morn"].to_numpy(dtype=float) - c_star) ** 2 + (daily["val_even"].to_numpy(dtype=float) - c_star) ** 2
        RMSD = float(np.sqrt(np.sum(diffsq) / N))
        RMSD_over_c = float(RMSD / c_star) if c_star != 0 else np.nan

        rows_rmsd.append({"series": col, "N_days": N, "mean_morning": mean_m, "mean_evening": mean_e,
                          "c_star": c_star, "RMSD": RMSD, "RMSD_over_c_star": RMSD_over_c})

    df_rmsd = pd.DataFrame(rows_rmsd).sort_values("series")
    try:
        from openpyxl import load_workbook  # if available, append as a new sheet
        with pd.ExcelWriter(EXCEL_BY_ZONES, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
            df_rmsd.to_excel(w, sheet_name="RMSD_ON_morn_even", index=False)
    except Exception:
        # fallback: recreate workbook with RMSD only
        with pd.ExcelWriter(EXCEL_BY_ZONES, engine="xlsxwriter") as w:
            for zname, d in zone_tables.items():
                d.to_excel(w, sheet_name=zname, index=False)
            df_rmsd.to_excel(w, sheet_name="RMSD_ON_morn_even", index=False)

    print("Done.")
    print(f"[Excel] All-year percentiles: {EXCEL_ALL_YEAR}")
    print(f"[PDF]   Box plots (all series, one page): {PDF_BOX_ALL_YEAR}")
    print(f"[Excel] Zone-based stats (+ RMSD sheet): {EXCEL_BY_ZONES}")
    print(f"[PDF]   Box plots by zone (grid): {PDF_BOX_ZONES}")


Done.
[Excel] All-year percentiles: C:\Users\admin\load_profiles_2024_r2\subclass_reports\distribution_outputs_bldg\bldg_all_year_percentiles.xlsx
[PDF]   Box plots (all series, one page): C:\Users\admin\load_profiles_2024_r2\subclass_reports\distribution_outputs_bldg\boxplots_all_series_onepage.pdf
[Excel] Zone-based stats (+ RMSD sheet): C:\Users\admin\load_profiles_2024_r2\subclass_reports\distribution_outputs_bldg\bldg_timezones_statistics.xlsx
[PDF]   Box plots by zone (grid): C:\Users\admin\load_profiles_2024_r2\subclass_reports\distribution_outputs_bldg\boxplots_by_zone_grid_onepage.pdf
